# SignaVis Python Demo (Hero)

This notebook shows the compact 'hero' player variant and avoids the IPython HTML/srcdoc warning by using an IFrame data-URL.

In [ ]:
!pip install signavis

In [ ]:
import requests, re
from pathlib import Path

audio_url = 'https://xeno-canto.org/1089065/download'
r = requests.get(audio_url)
ext = None
cd = r.headers.get('Content-Disposition', '')
match = re.search(r'filename="?(.+?)"?$', cd)
if match:
    ext = Path(match.group(1)).suffix
if not ext:
    import mimetypes
    content_type = r.headers.get('Content-Type', '')
    if 'audio/' in content_type:
        ext = mimetypes.guess_extension(content_type.split(';')[0]) or None
if not ext:
    ext = '.mp3'
audio_path = f'sample_audio{ext}'
with open(audio_path, 'wb') as f:
    f.write(r.content)
print('Downloaded', audio_path)

In [ ]:
import IPython.display as ipd
ipd.Audio(audio_path)

In [ ]:
from signavis import render_daw_player
from IPython.display import IFrame, HTML
import base64, re, html as _html

with open(audio_path, 'rb') as f:
    audio_bytes = f.read()

html = render_daw_player(
    audio_bytes,
    viewMode='spectrogram',
    transportStyle='hero',
    iframe_height=400,
    showFileOpen=False,
    showFFTControls=False,
    showOverview=False,
    showStatusbar=False,
    showDisplayGain=False,
    showViewToggles=False,
    showZoom=False,
    transportOverlay=True
)

m = re.search(r'srcdoc="(.*)"', html, flags=re.DOTALL)
if m:
    escaped = m.group(1)
    srcdoc = _html.unescape(escaped)
    data_url = 'data:text/html;base64,' + base64.b64encode(srcdoc.encode('utf-8')).decode('ascii')
    display = IFrame(src=data_url, width='100%', height=400)
    display
else:
    HTML(html)